In [1]:
import mlflow
import joblib
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from azureml.core import Workspace
import warnings

warnings.filterwarnings('ignore')

/Users/alfarouq/Library/CloudStorage/OneDrive-UniversitéMohammedVIPolytechnique/S9/cloud computing/lab2/Lab Folder MLOps/.venv/lib/python3.13/site-packages/azureml/core/__init__.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
ws = Workspace.from_config()

mlflow_tracking_uri = ws.get_mlflow_tracking_uri()
mlflow.set_tracking_uri(mlflow_tracking_uri)

In [2]:
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
scaler_filename = "scaler.joblib"
joblib.dump(scaler, scaler_filename)

['scaler.joblib']

In [3]:
mlflow.set_experiment("Breast_Cancer_Deployment")

with mlflow.start_run(run_name="Azure_LogisticRegression") as run:
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("solver", "liblinear")
    
    model_lr = LogisticRegression(C=1.0, solver='liblinear', random_state=42)
    model_lr.fit(X_train_scaled, y_train)

    artifact_path = "model_files"
    mlflow.sklearn.log_model(model_lr, artifact_path)
    mlflow.log_artifact(scaler_filename, artifact_path)
    
    y_pred = model_lr.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred) 
    
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc", auc)
    
    model_uri = f"runs:/{run.info.run_id}/{artifact_path}"
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name="breast-cancer-model-v2"  
    )
    
    print(f"Model trained with AUC: {auc:.4f}")
    print(f"Model logged to Azure in run: {run.info.run_id}")
    print(f"Model registered as: {registered_model.name} (Version: {registered_model.version})")

2025/11/17 20:42:59 INFO mlflow.tracking.fluent: Experiment with name 'Breast_Cancer_Deployment' does not exist. Creating a new experiment.
2025/11/17 20:43:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/17 20:43:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Model trained with AUC: 0.9812
Model logged to Azure in run: b99cbb4d4e504ba18263da2e0bcc1e9e
Model registered as: breast-cancer-model-v2 (Version: 1)


Successfully registered model 'breast-cancer-model-v2'.
Created version '1' of model 'breast-cancer-model-v2'.


In [ ]:
# %%
from azureml.core.webservice import AciWebservice
from azureml.core import Model
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.model import InferenceConfig

azure_model = Model(ws, "breast-cancer-model-v2")

aci_config = AciWebservice.deploy_configuration(
    cpu_cores=1,
    memory_gb=1,
    description="Breast Cancer API",
)

print("Creating environment...")
env = Environment("cancer-model-env-v2") 
conda_dep = CondaDependencies()

conda_dep.add_pip_package("joblib")
conda_dep.add_pip_package("pandas")
conda_dep.add_pip_package("scikit-learn")
conda_dep.add_pip_package("azureml-defaults") 

env.python.conda_dependencies = conda_dep
env.register(workspace=ws)
print(f"Environment {env.name} registered.")

inference_config = InferenceConfig(
    entry_script="score.py",
    environment=env
)

service_name = "cancer-service-aci-v2"
print(f"Deploying {service_name}...")

service = Model.deploy(
    workspace=ws,
    name=service_name,
    models=[azure_model], 
    inference_config=inference_config, 
    deployment_config=aci_config,
    overwrite=True
)

service.wait_for_deployment(show_output=True)

print(f"Deployment complete.")
print(f"API Endpoint (scoring_uri): {service.scoring_uri}")

Creating environment...
Environment cancer-model-env-v2 registered.
Deploying cancer-service-aci-v2...
Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2025-11-17 20:38:02+01:00 Creating Container Registry if not exists.
2025-11-17 20:38:03+01:00 Use the existing image.
2025-11-17 20:38:04+01:00 Generating deployment configuration.
2025-11-17 20:38:07+01:00 Submitting deployment to compute.
Failed


Service deployment polling reached non-successful terminal state, current service state: Unhealthy
Operation ID: 4fe6ab13-c7ee-4be8-990d-8253ca43dc79
More information can be found using '.get_logs()'
Error:
{
  "code": "AuthorizationFailed",
  "statusCode": 403,
  "message": "ACI Service request failed. Reason: The client '80d6744a-25bc-4f8e-bc76-d4f1d94757b9' with object id '3573cdf5-18e9-4502-b00e-a83c037f9a13' does not have authorization to perform action 'Microsoft.ContainerInstance/containerGroups/write' over scope '/subscriptions/0f8b334a-b3da-4977-9703-f11ffc829e63/resourceGroups/Learn-MLOps/providers/Microsoft.ContainerInstance/containerGroups/cancer-service-aci-v2-JMctwlNI9E_ul2RjEHXI2A' or the scope is invalid. If access was recently granted, please refresh your credentials.."
}



WebserviceException: WebserviceException:
	Message: Service deployment polling reached non-successful terminal state, current service state: Unhealthy
Operation ID: 4fe6ab13-c7ee-4be8-990d-8253ca43dc79
More information can be found using '.get_logs()'
Error:
{
  "code": "AuthorizationFailed",
  "statusCode": 403,
  "message": "ACI Service request failed. Reason: The client '80d6744a-25bc-4f8e-bc76-d4f1d94757b9' with object id '3573cdf5-18e9-4502-b00e-a83c037f9a13' does not have authorization to perform action 'Microsoft.ContainerInstance/containerGroups/write' over scope '/subscriptions/0f8b334a-b3da-4977-9703-f11ffc829e63/resourceGroups/Learn-MLOps/providers/Microsoft.ContainerInstance/containerGroups/cancer-service-aci-v2-JMctwlNI9E_ul2RjEHXI2A' or the scope is invalid. If access was recently granted, please refresh your credentials.."
}
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "Service deployment polling reached non-successful terminal state, current 